<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-04-rag/lesson-4.2-diy-rag/notebooks/GCP_Capstone_4.2_DIY_RAG.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.2 DIY RAG Pipeline — Retrieve → Augment → Generate
**Netsetos GenAI Engineering — GCP Capstone**

Complete RAG: embed query, retrieve vectors, augment prompt, generate with citations, track cost.


## Setup


In [ ]:
!pip install -q google-genai google-cloud-firestore pydantic
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google import genai
from google.genai import types
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from pydantic import BaseModel, Field
from typing import List, Literal
import json, subprocess

client = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')  # embeddings: regional only
gen_client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')   # Gemini 3.x generation: global only
db = firestore.Client(project=PROJECT_ID)

# --- Resources this lesson queries (rag_chunks): DB + vector index + seed data ---
# 1) Firestore (default) database (idempotent)
if '(default)' not in subprocess.run(
        ['gcloud', 'firestore', 'databases', 'list', '--project', PROJECT_ID, '--format=value(name)'],
        capture_output=True, text=True).stdout:
    print('Creating Firestore (default) database...')
    subprocess.run(['gcloud', 'firestore', 'databases', 'create',
                    '--location=asia-south1', '--project', PROJECT_ID], check=False)

# 2) Vector index on rag_chunks.embedding (768-dim) for find_nearest (idempotent)
_idx = subprocess.run(
    "gcloud firestore indexes composite create --project=" + PROJECT_ID +
    " --collection-group=rag_chunks --query-scope=COLLECTION"
    " --field-config=vector-config='{\"dimension\":\"768\",\"flat\":\"{}\"}',field-path=embedding",
    shell=True, capture_output=True, text=True)
_out = (_idx.stdout + _idx.stderr).lower()
print('rag_chunks vector index:',
      'creating (~2-5 min to build)' if _idx.returncode == 0
      else ('already exists' if 'already exists' in _out else (_idx.stderr.strip()[:90])))

# 3) Seed a few EMBEDDED chunks so retrieve returns results (skipped if already present)
if not list(db.collection('rag_chunks').limit(1).stream()):
    print('Seeding rag_chunks with sample embedded chunks...')
    for _i, (_txt, _src) in enumerate([
            ('RAG combines retrieval with generation to ground LLM answers in your own documents.', 'rag_intro.pdf'),
            ('Vector search finds semantically similar chunks using cosine distance over embeddings.', 'vector_search.pdf'),
            ('Chunking splits documents into passages sized for the model context window.', 'chunking.pdf'),
            ('Citations map each answer sentence back to its source chunk for auditability.', 'citations.pdf'),
            ('Firestore find_nearest does native KNN vector search, no separate vector database.', 'firestore_vector.pdf')]):
        _v = client.models.embed_content(
            model='text-embedding-005', contents=_txt,
            config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT', output_dimensionality=768)
        ).embeddings[0].values
        db.collection('rag_chunks').document(f'seed_{_i}').set(
            {'content': _txt, 'source_file': _src, 'pages': '1', 'embedding': Vector(_v)})
    print('Seeded 5 chunks. NOTE: the vector index must reach READY (~2-5 min) before Cell 2 find_nearest works.')
else:
    print('rag_chunks already has data.')

## Cell 1: Stage 1 — Embed the Query


In [ ]:
def embed_query(query):
    result = client.models.embed_content(
        model='text-embedding-005', contents=query,
        config=types.EmbedContentConfig(
            task_type='RETRIEVAL_QUERY',
            output_dimensionality=768))
    return result.embeddings[0].values

qv = embed_query('What are the latest trends in RAG?')
print(f'Query vector: {len(qv)} dims, first 5: {qv[:5]}')


## Cell 2: Stage 2 — Retrieve from Firestore


In [ ]:
def retrieve_chunks(query_vector, collection='rag_chunks', top_k=5):
    results = db.collection(collection).find_nearest(
        vector_field='embedding', query_vector=Vector(query_vector),
        distance_measure=DistanceMeasure.COSINE,
        limit=top_k, distance_threshold=0.3).get()
    chunks = []
    for doc in results:
        data = doc.to_dict()
        chunks.append({
            'chunk_id': doc.id,
            'content': data.get('content',''),
            'source': data.get('source_file','unknown'),
            'pages': data.get('pages',''),
        })
    return chunks

chunks = retrieve_chunks(qv)
print(f'Retrieved {len(chunks)} chunks')
for c in chunks:
    print(f'  {c["source"]} | {c["content"][:60]}...')


## Cell 3: Stage 3 — Augment Prompt


In [ ]:
def build_rag_prompt(query, chunks):
    if not chunks:
        return f'Question: {query}\n\nNo relevant context found.'
    ctx = '\n\n'.join(
        f'[Source {i+1}] ({c["source"]}, pages {c["pages"]})\n{c["content"]}'
        for i, c in enumerate(chunks))
    return f'Context:\n{ctx}\n\nQuestion: {query}'

prompt = build_rag_prompt('What are trends in RAG?', chunks)
print(prompt[:400])


## Cell 4: Stage 4 — Generate with Citations


In [ ]:
class Citation(BaseModel):
    source_id: int = Field(description='Source number [1-N]')
    chunk_id: str = Field(description='Firestore doc ID')
    relevance: float

class RAGResponse(BaseModel):
    answer: str = Field(description='Answer from context only')
    confidence: Literal['high','medium','low']
    citations: List[Citation]
    needs_more_context: bool

RAG_SYSTEM = '''You are DocuMind. Answer ONLY from context.
Cite every claim with [Source N]. Rate confidence.'''

def generate_rag(prompt, chunks):
    chunk_map = {i+1: c['chunk_id'] for i, c in enumerate(chunks)}
    r = gen_client.models.generate_content(
        model='gemini-3.6-flash', contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=RAG_SYSTEM,
            response_mime_type='application/json',
            response_schema=RAGResponse,
            temperature=0.1,
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    result = r.parsed
    for c in result.citations:
        c.chunk_id = chunk_map.get(c.source_id, 'unknown')
    return result, r.usage_metadata

result, usage = generate_rag(prompt, chunks)
print(f'Answer: {result.answer[:200]}...')
print(f'Confidence: {result.confidence}')
print(f'Citations: {len(result.citations)}')


## Cell 5: Stage 5 — Cost Tracking


In [ ]:
def calc_cost(usage, n_chunks=5):
    inp = usage.prompt_token_count
    out = usage.candidates_token_count
    think = usage.thoughts_token_count or 0
    cost = inp/1e6*1.50 + (out+think)/1e6*7.50 + 0.00001
    print(f'Input: {inp} tokens (${inp/1e6*1.50:.6f})')
    print(f'Output: {out} tokens (${out/1e6*7.50:.6f})')
    print(f'Thinking: {think} tokens')
    print(f'Total: ${cost:.6f} (Rs {cost*85:.4f})')
    return cost

calc_cost(usage)


## Cell 6: Complete Pipeline — ask_documind()


In [ ]:
def ask_documind(query, top_k=5):
    qv = embed_query(query)
    chunks = retrieve_chunks(qv, top_k=top_k)
    prompt = build_rag_prompt(query, chunks)
    if not chunks:
        return {'answer':'No relevant context.','confidence':'low','citations':[],'cost':0}
    result, usage = generate_rag(prompt, chunks)
    cost = usage.prompt_token_count/1e6*1.50 + usage.candidates_token_count/1e6*7.50
    return {
        'answer': result.answer,
        'confidence': result.confidence,
        'citations': [c.model_dump() for c in result.citations],
        'needs_more': result.needs_more_context,
        'cost_usd': cost}

r = ask_documind('What is RAG?')
print(f'Answer: {r["answer"][:150]}...')
print(f'Confidence: {r["confidence"]}')
print(f'Citations: {len(r["citations"])}')
print(f'Cost: ${r["cost_usd"]:.6f}')


## Cell 7: Citation Verification


In [ ]:
def verify_citations(rag_result, chunks):
    valid_ids = set(range(1, len(chunks)+1))
    issues = []
    for c in rag_result.citations:
        if c.source_id not in valid_ids:
            issues.append(f'Invalid source_id: {c.source_id}')
    sentences = rag_result.answer.split('. ')
    uncited = sum(1 for s in sentences if '[Source' not in s)
    if uncited > 1:
        issues.append(f'{uncited} uncited sentences')
    return {'valid': len(issues)==0, 'issues': issues}

v = verify_citations(result, chunks)
print(f'Valid: {v["valid"]}')
for issue in v['issues']:
    print(f'  Issue: {issue}')


## Cell 8: RAGEngine Production Class


In [ ]:
class RAGEngine:
    def __init__(self, project, location='us-central1'):
        self.client = genai.Client(enterprise=True, project=project, location=location)      # embeddings: regional
        self.gen_client = genai.Client(enterprise=True, project=project, location='global')  # generation: global
        self.db = firestore.Client(project=project)
        self.total_cost = 0.0
        self.query_count = 0

    def query(self, question, collection='rag_chunks', top_k=5):
        qv = self.client.models.embed_content(
            model='text-embedding-005', contents=question,
            config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY',
                                            output_dimensionality=768)
        ).embeddings[0].values
        docs = self.db.collection(collection).find_nearest(
            vector_field='embedding', query_vector=Vector(qv),
            distance_measure=DistanceMeasure.COSINE,
            limit=top_k, distance_threshold=0.3).get()
        chunks = [{'id':d.id,'content':d.to_dict().get('content',''),
                   'source':d.to_dict().get('source_file','')} for d in docs]
        if not chunks:
            return {'answer':'No relevant context.','confidence':'low'}
        ctx = '\n\n'.join(f'[Source {i+1}]\n{c["content"]}' for i,c in enumerate(chunks))
        r = self.gen_client.models.generate_content(
            model='gemini-3.6-flash', contents=f'Context:\n{ctx}\n\nQuestion: {question}',
            config=types.GenerateContentConfig(
                system_instruction='Answer from context only. Cite [Source N].',
                response_mime_type='application/json', response_schema=RAGResponse,
                temperature=0.1, thinking_config=types.ThinkingConfig(thinking_budget=0)))
        cost = r.usage_metadata.prompt_token_count/1e6*1.50 + r.usage_metadata.candidates_token_count/1e6*7.50
        self.total_cost += cost
        self.query_count += 1
        return {'result':r.parsed, 'cost':cost, 'chunks':len(chunks)}

    def report(self):
        avg = self.total_cost/self.query_count if self.query_count else 0
        print(f'Queries: {self.query_count} | Total: ${self.total_cost:.4f} | Avg: ${avg:.6f}/q')

print('RAGEngine ready')


## ✅ Lesson 4.2 Complete!

- ✅ 5-stage RAG: embed → retrieve → augment → generate → return
- ✅ RETRIEVAL_QUERY vs RETRIEVAL_DOCUMENT task types
- ✅ Firestore find_nearest() with distance_threshold quality gate
- ✅ Numbered [Source N] context for citation tracking
- ✅ RAGResponse Pydantic schema with answer + citations + confidence
- ✅ Per-query cost tracking in USD and INR
- ✅ Citation verification and hallucination detection
- ✅ RAGEngine production module

**Next: Lesson 4.3 — Advanced RAG (HyDE, re-ranking, hybrid search)**
